In [51]:
from typing import TypedDict, Literal
from pydantic import BaseModel, Field

from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from IPython.display import Image, display

In [52]:
load_dotenv()

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
    max_tokens=250
)

In [53]:
class Sentiment(BaseModel):
    sentiment: Literal["positive", "negative"]


class Diagnosis(BaseModel):
    issue_type: str = Field(description="Main problem in 2-4 words")
    tone: str = Field(description="Customer's emotional tone")
    urgency: Literal["low", "medium", "high"]


sentiment_llm = llm.with_structured_output(Sentiment, method="function_calling")
diagnosis_llm = llm.with_structured_output(Diagnosis, method="function_calling")

Let's use `Literal` for the Pydantic structured output fields wherever the possible values are predefined, similar to CampusX's implementation. Keep the existing workflow unchanged and update only the schema.

For example:

```python
class Sentiment(BaseModel):
    sentiment: Literal["positive", "negative"]

class Diagnosis(BaseModel):
    issue_type: Literal["delivery", "product_quality", "customer_service", "other"]
    tone: Literal["angry", "frustrated", "disappointed", "neutral"]
    urgency: Literal["low", "medium", "high"]
```

Use `Literal` to restrict the LLM to predefined categories rather than allowing arbitrary strings. Keep the code simple and beginner-friendly.

In [54]:
class ReviewState(TypedDict):
    review: str
    sentiment: str
    issue_type: str
    tone: str
    urgency: str
    response: str

In [62]:
def find_sentiment(state: ReviewState):
    response = llm.invoke(
        f"Classify this review as positive or negative. "
        f"Reply with only one word: positive or negative.\n"
        f"Review: {state['review']}"
    )

    sentiment = response.content.strip().lower()

    return {"sentiment": sentiment}

In [63]:
def run_diagnosis(state: ReviewState):
    result = diagnosis_llm.invoke(
        f"Identify the issue type, tone, and urgency of this review:\n{state['review']}"
    )

    return result.model_dump()

In [64]:
def positive_response(state: ReviewState):
    reply = llm.invoke(
        f"Write a friendly, one-sentence reply to this positive review:\n{state['review']}"
    )

    return {"response": reply.content}

In [65]:
def negative_response(state: ReviewState):
    prompt = f"""
    Write a short, empathetic customer support reply in 2 sentences.

    Review: {state['review']}
    Issue: {state['issue_type']}
    Tone: {state['tone']}
    Urgency: {state['urgency']}

    Don't promise refunds or actions that aren't confirmed.
    """

    reply = llm.invoke(prompt)

    return {"response": reply.content}

In [66]:
def check_sentiment(state: ReviewState) -> Literal["positive_response", "run_diagnosis"]:
    if state["sentiment"] == "positive":
        return "positive_response"
    else:
        return "run_diagnosis"

In [67]:
graph = StateGraph(ReviewState)

graph.add_node("find_sentiment", find_sentiment)
graph.add_node("run_diagnosis", run_diagnosis)
graph.add_node("positive_response", positive_response)
graph.add_node("negative_response", negative_response)

graph.add_edge(START, "find_sentiment")

graph.add_conditional_edges(
    "find_sentiment",
    check_sentiment
)

graph.add_edge("run_diagnosis", "negative_response")

graph.add_edge("positive_response", END)
graph.add_edge("negative_response", END)

workflow = graph.compile()

In [69]:
initial_state = {
    "review": "My order arrived three days late and the product was damaged. Very disappointed."
}

final_state = workflow.invoke(initial_state)

print("Sentiment:", final_state["sentiment"])

if final_state["sentiment"] == "negative":
    print("Issue:", final_state["issue_type"])
    print("Tone:", final_state["tone"])
    print("Urgency:", final_state["urgency"])

print("\nResponse:", final_state["response"])

Sentiment: negative
Issue: Late delivery and product damage
Tone: disappointed
Urgency: high

Response: I’m really sorry to hear that your order arrived late and was damaged—this is definitely not the experience we aim to provide. We’re looking into the issue right now and will keep you updated on the next steps as soon as we have more information.
